# Download dataset

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews   -p .

## Import necessary libraries

In [14]:
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import torch.nn.init as init

import nltk
from collections import Counter
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
# nltk.download('punkt')
# nltk.download('stopwords')
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [ ]:
with zipfile.ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
    zip_ref.extractall(".")  

# Preprocessing:
- Lower words, remove all punctuations and tags, remove extra spaces
- Remove stopwords and stemming
- Extract to a `cleaned_IMDB_Dataset.csv` file for checkpoint

In [ ]:
df =pd.read_csv("IMDB Dataset.csv")

import string
import re
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'<br\s*/?>', ' ', text)  
    text = text.translate(str.maketrans(' ', ' ', string.punctuation))  
    text = re.sub(r'\s+', ' ', text).strip() 
    return text
def remove_stopwords_and_stem(text):
    stop_words = set(stopwords.words('english'))  
    tokenizer = TreebankWordTokenizer()  
    stemmer = PorterStemmer()  

    words = tokenizer.tokenize(text)  
    filtered_words = [
        stemmer.stem(word) for word in words if word.lower() not in stop_words
    ]  
    
    return ' '.join(filtered_words)

df["review"] = df["review"].apply(preprocess_text)
df["review"] = df["review"].apply(remove_stopwords_and_stem)
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

print(df["review"][0])
print(df["label"].value_counts())

df.to_csv("cleaned_IMDB_Dataset.csv", index=False)


- Split the dataset to 90% first as training set and 10% last as testing set

In [17]:
df =pd.read_csv("cleaned_IMDB_Dataset.csv")

split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]
test_df.head()

,review,sentiment,label
45000,enjoy film sceneri corfu greek ador countri li...,positive,1
45001,macarthur great movi great stori great man gen...,positive,1
45002,say ignor review went see damn review right wa...,negative,0
45003,pretti transpar attempt wring cash thrive brit...,negative,0
45004,even though book wasnt strictli accur real sit...,negative,0


- Count and sort the most common words for indexing

In [18]:
tokenizer = TreebankWordTokenizer()
counter=Counter()
for text in train_df["review"]:
    counter.update(tokenizer.tokenize(text))

In [19]:
vocab = {word: i + 2 for i, (word, _) in enumerate(counter.most_common())}
vocab["<unk>"] = 0
vocab["<pad>"] = 1

print(list(vocab.items())[:20])

[('movi', 2), ('film', 3), ('one', 4), ('like', 5), ('time', 6), ('good', 7), ('make', 8), ('charact', 9), ('see', 10), ('get', 11), ('watch', 12), ('even', 13), ('stori', 14), ('would', 15), ('realli', 16), ('scene', 17), ('well', 18), ('show', 19), ('look', 20), ('much', 21)]


### Prepare dataloader

In [20]:
class IMDBDataset(Dataset):
    def __init__(self, df, vocab):
        self.texts = df["review"].tolist()
        self.labels = df["label"].tolist()
        self.vocab = vocab
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = tokenizer.tokenize(self.texts[idx])
        token_ids = [self.vocab.get(token, self.vocab["<unk>"]) for token in tokens]
        label = float(self.labels[idx])  
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float)


In [21]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.tensor(labels, dtype=torch.float)  
    return texts.to(device), labels.to(device)


train_dataset = IMDBDataset(train_df, vocab)
test_dataset = IMDBDataset(test_df, vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)


## Model definitions:
- Two hidden layers with dropout
- Add dropout after embedding and before the last layer to address overfitting
- Add xavier initialization and batchnorm to stabilize training

In [29]:
import torch.nn.init as init

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, rnn_type="gru"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab["<pad>"])
        self.dropout_embedding=nn.Dropout(0.3)
        self.dropout_fc=nn.Dropout(0.3)
        if rnn_type == "rnn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, num_layers=2,dropout=0.3)
        elif rnn_type == "gru":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True, num_layers=2,dropout=0.3)
        else:
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2,dropout=0.3)

        # Initialize the weights
        for name, param in self.rnn.named_parameters():
            if 'weight_ih' in name:
                init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                init.orthogonal_(param)
            elif 'bias' in name:
                init.zeros_(param)
                
        self.batchnorm = nn.BatchNorm1d(hidden_dim) 
        self.fc = nn.Linear(hidden_dim, output_dim)
        init.xavier_uniform_(self.fc.weight)
        init.zeros_(self.fc.bias)
    
    def forward(self, x):
        embedded = self.embedding(x)  # Shape: (batch, seq_len, embed_dim)
        embedded=self.dropout_embedding(embedded)
        rnn_out, _ = self.rnn(embedded)  # Shape: (batch, seq_len, hidden_dim)

        last_hidden = rnn_out[:, -1, :]  # Last timestep output (batch, hidden_dim)  
        last_hidden = self.batchnorm(last_hidden)  
        last_hidden = self.dropout_fc(last_hidden)  

        out = self.fc(last_hidden)
        return torch.sigmoid(out).squeeze()




- Use Adam optimizer.
- For each epoch: train and update loss, evaluate accuracy on test set and print it out to keep track

In [ ]:
def train_and_evaluate(rnn_type, hidden_dim, lr, epochs):
    model = RNNModel(len(vocab), embed_dim=128, hidden_dim=hidden_dim, output_dim=1, rnn_type=rnn_type).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    accuracy = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        # Training phase
        for texts, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()

        # Evaluation phase
        model.eval()
        correct = 0
        predict_true = 0
        total = 0
        val_loss = 0.0

        with torch.no_grad():
            for texts, labels in test_loader:
                outputs = model(texts)
                loss = criterion(outputs, labels)
                val_loss += loss.item()  # Summing the validation loss
                predicted = (outputs > 0.5).float()
                predict_true += (predicted == 1.0).sum().item()
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        accuracy = correct / total
        avg_val_loss = val_loss / len(test_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}, Accuracy: {accuracy:.4f}, NumPositive: {predict_true}, Validation Loss: {avg_val_loss:.4f}")

        # scheduler.step(avg_val_loss)  # Pass validation loss to the scheduler

    print(f"{rnn_type.upper()} Accuracy: {accuracy:.4f}")
    return accuracy


In [24]:
rnn_acc = train_and_evaluate("rnn",64,0.001,50)

Epoch 1, Loss: 0.7373, Accuracy: 0.4968, NumPositive: 2478, Validation Loss: 0.6954
Epoch 2, Loss: 0.6973, Accuracy: 0.5064, NumPositive: 2558, Validation Loss: 0.6931
Epoch 3, Loss: 0.6945, Accuracy: 0.4976, NumPositive: 2542, Validation Loss: 0.6942
Epoch 4, Loss: 0.6946, Accuracy: 0.5164, NumPositive: 2552, Validation Loss: 0.6930
Epoch 5, Loss: 0.6941, Accuracy: 0.5032, NumPositive: 2522, Validation Loss: 0.6935
Epoch 6, Loss: 0.6941, Accuracy: 0.4978, NumPositive: 2487, Validation Loss: 0.6934
Epoch 7, Loss: 0.6940, Accuracy: 0.4938, NumPositive: 2511, Validation Loss: 0.6945
Epoch 8, Loss: 0.6927, Accuracy: 0.5012, NumPositive: 2522, Validation Loss: 0.6934
Epoch 9, Loss: 0.6886, Accuracy: 0.5680, NumPositive: 668, Validation Loss: 0.6756
Epoch 10, Loss: 0.6724, Accuracy: 0.6226, NumPositive: 3127, Validation Loss: 0.6640
Epoch 11, Loss: 0.6601, Accuracy: 0.6674, NumPositive: 2309, Validation Loss: 0.6382
Epoch 12, Loss: 0.6493, Accuracy: 0.6682, NumPositive: 2985, Validation Los

In [32]:
gru_acc = train_and_evaluate("gru",128,0.001,25)

Epoch 1, Loss: 0.7502, Accuracy: 0.4960, NumPositive: 4944, Validation Loss: 0.6934
Epoch 2, Loss: 0.7016, Accuracy: 0.5066, NumPositive: 265, Validation Loss: 0.6948
Epoch 3, Loss: 0.6745, Accuracy: 0.8224, NumPositive: 2398, Validation Loss: 0.4197
Epoch 4, Loss: 0.3574, Accuracy: 0.8826, NumPositive: 2553, Validation Loss: 0.2853
Epoch 5, Loss: 0.2556, Accuracy: 0.8972, NumPositive: 2472, Validation Loss: 0.2607
Epoch 6, Loss: 0.2090, Accuracy: 0.8970, NumPositive: 2335, Validation Loss: 0.2689
Epoch 7, Loss: 0.1757, Accuracy: 0.8960, NumPositive: 2568, Validation Loss: 0.2880
Epoch 8, Loss: 0.1492, Accuracy: 0.9008, NumPositive: 2562, Validation Loss: 0.2801
Epoch 9, Loss: 0.1277, Accuracy: 0.9046, NumPositive: 2399, Validation Loss: 0.3061
Epoch 10, Loss: 0.1111, Accuracy: 0.8984, NumPositive: 2642, Validation Loss: 0.3262
Epoch 11, Loss: 0.0944, Accuracy: 0.9008, NumPositive: 2590, Validation Loss: 0.3331
Epoch 12, Loss: 0.0888, Accuracy: 0.8948, NumPositive: 2354, Validation Los

In [33]:
lstm_acc = train_and_evaluate("lstm",128,0.001,25)

Epoch 1, Loss: 0.7426, Accuracy: 0.4936, NumPositive: 4924, Validation Loss: 0.7029
Epoch 2, Loss: 0.6983, Accuracy: 0.4976, NumPositive: 4922, Validation Loss: 0.6923
Epoch 3, Loss: 0.6947, Accuracy: 0.4960, NumPositive: 4924, Validation Loss: 0.6957
Epoch 4, Loss: 0.6930, Accuracy: 0.5090, NumPositive: 103, Validation Loss: 0.6963
Epoch 5, Loss: 0.6916, Accuracy: 0.5152, NumPositive: 126, Validation Loss: 0.6908
Epoch 6, Loss: 0.6907, Accuracy: 0.6328, NumPositive: 3226, Validation Loss: 0.6911
Epoch 7, Loss: 0.3827, Accuracy: 0.8816, NumPositive: 2372, Validation Loss: 0.2843
Epoch 8, Loss: 0.2567, Accuracy: 0.8966, NumPositive: 2467, Validation Loss: 0.2663
Epoch 9, Loss: 0.2103, Accuracy: 0.8982, NumPositive: 2575, Validation Loss: 0.2672
Epoch 10, Loss: 0.1708, Accuracy: 0.8950, NumPositive: 2695, Validation Loss: 0.2915
Epoch 11, Loss: 0.1485, Accuracy: 0.9020, NumPositive: 2580, Validation Loss: 0.2920
Epoch 12, Loss: 0.1287, Accuracy: 0.9020, NumPositive: 2442, Validation Loss

In [34]:
print(f"\nPerformance Comparison:\nRNN: {rnn_acc:.4f}\nGRU: {gru_acc:.4f}\nLSTM: {lstm_acc:.4f}")



Performance Comparison:
RNN: 0.7426
GRU: 0.9010
LSTM: 0.8958


## Conclusion:
Final accuracy indicates that GRU and LSTM are much better at processing long sequences since they do not suffer from vanishing gradient problem. Stopwords are removed to make the sequences shorter to help vanila RNN. Vanila RNN despite getting more epoches to train still could not surprass 75% accuracy. Although not shown in this notebook, adding residual connection to `nn.RNN` can help it significantly and increase the accuracy to 90%.